# OCR Bilans Fiscaux Algériens — V9 — H100 + Qwen3.6-VL-27B
Pipeline: PDF → JSON (ACTIF/PASSIF/TCR D-C/DECL) → Excel récap → FORFAIT .xlsx (rouge si incohérent).
Règle d'or: jamais de valeur inventée; absent/illisible = null + anomalie.

In [ ]:
%pip install -q -U "transformers>=4.57.0" accelerate pymupdf pillow openpyxl psutil pandas
print('✅ OK')

In [ ]:
import time, json, re, gc, difflib, csv, shutil, unicodedata
import numpy as np
from pathlib import Path
from datetime import datetime
from collections import defaultdict
import fitz, torch
from PIL import Image
from transformers import AutoProcessor, AutoModelForImageTextToText
from openpyxl import Workbook, load_workbook
from openpyxl.styles import Font, PatternFill, Alignment
from openpyxl.utils import get_column_letter
print('✅ Imports OK')

In [ ]:
MODEL_PATH='/domino/edv/modelhub/ModelHub-model-huggingface-Qwen/Qwen3.6-27B-FP8/main'
DEVICE='cuda' if torch.cuda.is_available() else 'cpu'
MAX_NEW_TOKENS=3000; IMAGE_MAX_SIZE=2024
MIN_PIXELS=4*32*32; MAX_PIXELS=2000*32*32
PDF_ZOOM=3.0; BLANK_THRESHOLD=0.95; GPU_BATCH_SIZE=8
ANNEE_ATTENDUE=None
TOL=100  # tolérance contrôles en dinars
INPUT_DIR=Path('/mnt/Risk/bilans_in')
OUTPUT_DIR=Path('/mnt/Risk/bilans_out')
JSON_DIR=OUTPUT_DIR/'json_bilans'
LOG_PATH=OUTPUT_DIR/'pipeline_bilans.log'
EXCEL_PATH=OUTPUT_DIR/f'bilans_{datetime.now().strftime("%Y%m%d_%H%M")}.xlsx'
TEMPLATE_FORFAIT=Path('/mnt/Risk/forfait/PREREMPLISSAGE_BILAN.xlsx')
FORFAIT_OUTPUT_DIR=OUTPUT_DIR/'forfaits'
for d in (OUTPUT_DIR,JSON_DIR,FORFAIT_OUTPUT_DIR): d.mkdir(parents=True,exist_ok=True)
pdfs=sorted(INPUT_DIR.glob('*.pdf'))
print(f'Device:{DEVICE} | PDF:{len(pdfs)} | Template:{TEMPLATE_FORFAIT.name}')

In [ ]:
t0=time.time()
processor=AutoProcessor.from_pretrained(MODEL_PATH,trust_remote_code=True,min_pixels=MIN_PIXELS,max_pixels=MAX_PIXELS)
processor.tokenizer.padding_side='left'
try:
    from transformers.integrations.finegrained_fp8 import FineGrainedFP8Config as FP8Config
except ImportError:
    from transformers import FineGrainedFP8Config as FP8Config
model=AutoModelForImageTextToText.from_pretrained(MODEL_PATH,dtype=torch.bfloat16,device_map='auto',trust_remote_code=True,low_cpu_mem_usage=True,quantization_config=FP8Config(dequantize=True))
model.eval()
print(f'✅ Modèle chargé en {time.time()-t0:.1f}s')

In [ ]:
def resize(img,max_side=IMAGE_MAX_SIZE):
    w,h=img.size
    if max(w,h)<=max_side: return img
    r=max_side/max(w,h); return img.resize((int(w*r),int(h*r)),Image.LANCZOS)
def estimate_skew(img):
    small=img.convert('L').copy(); small.thumbnail((500,500))
    def score(a):
        r=np.array(small.rotate(a,expand=True,fillcolor=255))<128
        return float(((r.sum(axis=1))**2).sum())
    best=max(range(-12,13,2),key=score)
    best=max([best-1,best-0.5,best,best+0.5,best+1],key=score)
    return best if abs(best)>=1 else 0.0
def deskew(img):
    a=estimate_skew(img)
    if a: img=img.rotate(a,expand=True,fillcolor=(255,255,255),resample=Image.BICUBIC)
    return img
def is_blank(image,threshold=BLANK_THRESHOLD):
    arr=np.array(image.convert('L')); return (arr>240).sum()/arr.size>=threshold
def pdf_to_pages(path,zoom=PDF_ZOOM):
    doc=fitz.open(path); matrix=fitz.Matrix(zoom,zoom); pages=[]
    for i in range(len(doc)):
        pix=doc.load_page(i).get_pixmap(matrix=matrix,alpha=False)
        pages.append({'index':i,'image':resize(deskew(Image.frombytes('RGB',[pix.width,pix.height],pix.samples)))})
    doc.close(); return pages
def parse_json(text):
    try:
        m=re.search(r'\{.*\}',text,re.S); return json.loads(m.group()) if m else {}
    except Exception: return {}
def apply_template(messages):
    try: return processor.apply_chat_template(messages,tokenize=False,add_generation_prompt=True,enable_thinking=False)
    except TypeError: return processor.apply_chat_template(messages,tokenize=False,add_generation_prompt=True)
def _decode(out_i,in_len): return processor.decode(out_i[in_len:],skip_special_tokens=True,clean_up_tokenization_spaces=False)
def ask_single(prompt,image):
    msgs=[{'role':'user','content':[{'type':'image','image':image},{'type':'text','text':prompt}]}]
    inputs=processor(text=[apply_template(msgs)],images=[image],return_tensors='pt').to(DEVICE)
    t0=time.time()
    with torch.no_grad():
        out=model.generate(**inputs,max_new_tokens=MAX_NEW_TOKENS,do_sample=False,repetition_penalty=1.0,pad_token_id=processor.tokenizer.eos_token_id)
    torch.cuda.synchronize()
    return {'text':_decode(out[0],inputs['input_ids'].shape[1]),'tokens_in':int(inputs['input_ids'].shape[1]),'tokens_out':int(out[0].shape[0]-inputs['input_ids'].shape[1]),'elapsed':round(time.time()-t0,2)}
def ask_batch(prompt,images):
    if not images: return []
    if len(images)==1: return [ask_single(prompt,images[0])]
    msgs=[[{'role':'user','content':[{'type':'image','image':img},{'type':'text','text':prompt}]}] for img in images]
    inputs=processor(text=[apply_template(m) for m in msgs],images=images,return_tensors='pt',padding=True).to(DEVICE)
    t0=time.time()
    with torch.no_grad():
        out=model.generate(**inputs,max_new_tokens=MAX_NEW_TOKENS,do_sample=False,repetition_penalty=1.0,pad_token_id=processor.tokenizer.eos_token_id)
    torch.cuda.synchronize(); el=time.time()-t0; in_len=inputs['input_ids'].shape[1]; attn=inputs.get('attention_mask')
    return [{'text':_decode(out[i],in_len),'tokens_in':int(attn[i].sum().item()) if attn is not None else in_len,'tokens_out':int(out[i].shape[0]-in_len),'elapsed':round(el/len(images),2)} for i in range(len(images))]
print('✅ Utilitaires OK')

In [ ]:
SCHEMAS={
 'ACTIF':{'cols':['brut','amort','n','n1'],'postes':{
   'ecarts_acquisition_goodwill':'Ecart d acquisition goodwill','immobilisations_incorporelles':'Immobilisations incorporelles','terrains':'Terrains','batiments':'Batiments','autres_immobilisations_corporelles':'Autres Immobilisations corporelles','immobilisations_en_concession':'Immobilisations en concession','immobilisations_en_cours':'Immobilisations en cours','titres_mis_en_equivalence':'Titres mis en equivalence','autres_participations_creances':'Autres participations et creances rattachees','autres_titres_immobilises':'Autres titres immobilises','prets_actifs_financiers_non_courants':'Prets et autres actifs financiers non courants','impots_differes_actif':'Impots Differes Actif','total_actif_non_courant':'TOTAL ACTIF NON COURANT','stocks_encours':'Stocks et encours','clients':'Clients','autres_debiteurs':'Autres debiteurs','impots_assimiles_actif':'Impots & Assimiles','autres_creances_assimiles':'Autres Creances & Emplois assimiles','placements_financiers_courants':'Placements et autres actifs financiers courants','tresorerie_actif':'Tresorerie','total_actif_courant':'TOTAL ACTIF COURANT','total_general_actif':'TOTAL GENERAL ACTIF'}},
 'PASSIF':{'cols':['n','n1'],'postes':{
   'capital_emis':'Capital emis','capital_non_appele':'Capital non appele','primes_reserves':'Primes et reserves','ecart_reevaluation':'Ecart de reevaluation','ecart_equivalence':'Ecart d equivalence','resultat_net_passif':'Resultat net','report_a_nouveau':'Report a nouveau','part_societe_consolidante':'Part de la societe consolidante','part_minoritaires':'Part des minoritaires','total_capitaux_propres':'TOTAL I','emprunts_dettes_financieres':'Emprunts et dettes financieres','impots_differes_provisionnes':'Impots differes et provisionnes','autres_dettes_non_courantes':'Autres dettes non courantes','provisions_produits_avance':'Provisions et produits comptabilises d avance','total_passifs_non_courants':'TOTAL PASSIFS NON COURANTS II','fournisseurs_rattaches':'Fournisseurs et comptes rattaches','impots_passif':'Impots','autres_dettes':'Autres dettes','tresorerie_passif':'Tresorerie Passif','total_passifs_courants':'TOTAL PASSIFS COURANTS','total_general_passif':'TOTAL GENERAL PASSIF'}},
 'TCR':{'cols':['debit_n','credit_n','debit_n1','credit_n1'],'postes':{
   'ventes_marchandises':'Ventes de Marchandises','produits_fabriques':'Produits Fabriques','prestations_services':'Prestations de Services','ventes_travaux':'Ventes de Travaux','produits_annexes':'Produits Annexes','rabais_remises_ristournes_accordes':'Rabais remises accordes','chiffre_affaires_net':'Chiffre d affaires net','production_stockee_destockee':'Production Stockee ou destockee','production_immobilisee':'Production immobilisee','subvention_exploitation':'Subvention d exploitation','production_exercice':'I-Production de l exercice','achats_marchandises_vendues':'Achats de Marchandises vendues','matieres_premieres':'Matieres premieres','autres_approvisionnements':'Autres Approvisionnements','variation_stocks':'Variation des Stocks','achats_etudes_prestations':'Achats d Etudes','autres_consommations':'Autres consommations','rabais_remises_obtenus_achats':'Rabais obtenus sur achats','sous_traitance_generale':'Sous-traitance generale','locations':'Locations','entretien_reparations':'Entretien reparations','primes_assurances':'Primes d assurances','personnel_exterieur':'Personnel exterieur','remuneration_intermediaires':'Remuneration d intermediaires','publicite':'Publicite','deplacements_missions':'Deplacements missions','autres_services':'Autres services','rabais_remises_obtenus_services':'Rabais obtenus services exterieurs','consommations_exercice':'II-Consommations de l exercice','valeur_ajoutee_exploitation':'III-Valeur ajoutee','charges_personnel':'Charges de personnel','impots_taxes_assimiles':'Impots et taxes','excedent_brut_exploitation':'IV-EBE','autres_produits_operationnels':'Autres produits operationnels','autres_charges_operationnelles':'Autres charges operationnelles','dotations_amortissements':'Dotations aux amortissements','provisions':'Provisions','pertes_valeur':'Perte de Valeur','reprises_pertes_valeur_provisions':'Reprise sur pertes de valeur','resultat_operationnel':'V-Resultat operationnel','produits_financiers':'Produits financiers','charges_financieres':'Charges financieres','resultat_financier':'VI-Resultat Financier','resultat_ordinaire':'VII-Resultat ordinaire','elements_extraordinaires_produits':'Elements extraordinaires Produits','elements_extraordinaires_charges':'Elements extraordinaires Charges','resultat_extraordinaire':'VIII-Resultat extraordinaire','impots_exigibles_resultats':'Impots exigibles sur resultats','impots_differes_resultats':'Impots differes sur resultats','resultat_net_exercice':'RESULTAT NET DE L EXERCICE'}},
 'DECL':{'cols':['valeur'],'postes':{
   'nif':'NIF','raison_sociale':'Raison sociale','activite_principale':'Activite principale','adresse_siege':'Adresse siege','registre_commerce':'Registre de Commerce','cac_cabinet':'Cabinet CAC','cac_nom':'Nom CAC','cac_agrement':'Agrement CAC','exercice_annee':'Annee exercice','exercice_periode_debut':'Periode debut','exercice_periode_fin':'Periode fin','annee_souscription':'Annee de souscription','chiffre_affaires_global_ht':'CA global HT','resultat_comptable':'Resultat comptable','resultat_fiscal':'Resultat fiscal'}}}
L=['Lis cette page d\'un dossier fiscal algerien (imprime Serie G).','',
'ETAPE 1 — type: ACTIF / PASSIF / TCR / DECL / AUTRE','',
'ETAPE 2 — JSON: {"type":...,"entreprise":...,"exercice":...,"nif":...,"annee_souscription":...,"postes":{cle:{col:montant}}}',
'- ACTIF: brut,amort,n,n1 | PASSIF: n,n1 | TCR: debit_n,credit_n,debit_n1,credit_n1 (une seule colonne remplie par poste)',
'- DECL: champs listes. Montant entre parentheses = negatif. NOMBRES JSON sans espaces. case vide=null.',
'- Utilise EXACTEMENT les cles ci-dessous. Si AUTRE: {"type":"AUTRE"}']
for t,spec in SCHEMAS.items():
    L.append(f'--- Si {t} ---')
    for k,lab in spec['postes'].items(): L.append(f'{k}: ligne "{lab}"')
L+=['','PRECISIONS SCAN: pages inclinees OK; cachets/tampons ignores (lire valeur imprimee); entreprise/nif/exercice en haut de chaque page.','REGLES: JSON valide seul, sans markdown, aucun champ invente.']
PROMPT_BILAN='\n'.join(L)
PROMPT_CLASSIF=('Page dun dossier fiscal algerien Serie G. Reponds UN seul mot: '
 'ACTIF si BILAN (ACTIF); PASSIF si BILAN (PASSIF); TCR si COMPTE DE RESULTAT; DECL si page DECLARATION; AUTRE sinon.')
PROMPT_BILAN+='\n- Dans postes, retourne UNIQUEMENT les cles avec valeur non nulle.'
print(f'✅ Schémas + prompt OK ({len(PROMPT_BILAN)} car.)')

In [ ]:
def norm_montant(v):
    if v is None: return None
    if isinstance(v,(int,float)): return float(v)
    s=str(v).strip(); neg=(s.startswith('(') and s.endswith(')')) or s.startswith('-')
    s=re.sub(r'[^\d.,]','',s)
    if not s: return None
    if s.count(',')==1 and '.' not in s: s=s.replace(',','.')
    elif ',' in s: s=s.replace(',','')
    elif s.count('.')>1: s=s.replace('.','')
    try: return -float(s) if neg else float(s)
    except Exception: return None
def norm_str(v):
    if v is None: return None
    s=re.sub(r'\s+',' ',str(v).strip())
    return s if s and s.lower() not in ('null','none','n/a') else None
def norm_nif(v):
    s=norm_str(v); return re.sub(r'[^0-9]','',s) if s else None
def norm_annee4(v):
    m=re.findall(r'20\d{2}',norm_str(v) or ''); return m[-1] if m else None
def normalise_table(table,data):
    spec=SCHEMAS[table]; raw=data.get('postes') or {}; out={}
    for key in spec['postes']:
        vals=raw.get(key); vals=vals if isinstance(vals,dict) else {}
        for col in spec['cols']: out[f'{key}_{col}']=norm_montant(vals.get(col))
    return out
def normalise_decl(data):
    raw=data.get('postes') or {}; d={}
    for k in SCHEMAS['DECL']['postes']:
        if k in ('nif',): d[k]=norm_nif(raw.get(k))
        elif k in ('exercice_annee','annee_souscription'): d[k]=norm_annee4(raw.get(k))
        elif k in ('chiffre_affaires_global_ht','resultat_comptable','resultat_fiscal'): d[k]=norm_montant(raw.get(k))
        else: d[k]=norm_str(raw.get(k))
    d['certifie_par_cac']=bool(d.get('cac_cabinet') or d.get('cac_nom'))
    return d
print('✅ Normalisation OK')

In [ ]:
def norm_identite(s):
    s=norm_upper(s) if (s:=norm_str(s)) else None
    return re.sub(r'[^A-Z0-9]','',s) if s else None
def norm_upper(v):
    s=norm_str(v); return s.upper() if s else None
def meme_entreprise(a,b):
    if not a or not b: return True
    if a==b or a in b or b in a: return True
    return difflib.SequenceMatcher(None,a,b).ratio()>0.80
def controle_coherence(parsed,annee_attendue=None):
    anomalies,excluded=[],set()
    idents=[(p['index'],norm_identite(p['meta'].get('entreprise'))) for p in parsed]
    vals=[e for _,e in idents if e]
    ref_ent=max(set(vals),key=lambda e:sum(1 for x in vals if meme_entreprise(e,x))) if vals else None
    for idx,e in idents:
        if ref_ent and e and not meme_entreprise(ref_ent,e): anomalies.append(f'PAGE {idx+1}: CLIENT DIFFERENT'); excluded.add(idx)
    annees=[(p['index'],norm_annee4(p['meta'].get('exercice'))) for p in parsed]
    yy=[a for _,a in annees if a]; ref_annee=max(set(yy),key=yy.count) if yy else None
    for idx,a in annees:
        if ref_annee and a and a!=ref_annee: anomalies.append(f'PAGE {idx+1}: ANNEE {a} ≠ {ref_annee}')
    if annee_attendue and ref_annee and ref_annee!=str(annee_attendue): anomalies.append(f'ANNEE DOSSIER {ref_annee} ≠ ATTENDUE {annee_attendue}')
    nifs=[(p['index'],norm_nif(p['meta'].get('nif'))) for p in parsed]
    nn=[n for _,n in nifs if n and len(n)>=12]; ref_nif=max(set(nn),key=nn.count) if nn else None
    for idx,n in nifs:
        if ref_nif and n and len(n)>=12 and n!=ref_nif: anomalies.append(f'PAGE {idx+1}: NIF DIFFERENT')
    return ref_ent,ref_nif,ref_annee,anomalies,excluded
print('✅ Cohérence OK')

In [ ]:
# ── FORMULES DE CONTRÔLE (sommes signées) ──
F_ACTIF=[('total_actif_non_courant',['ecarts_acquisition_goodwill','immobilisations_incorporelles','terrains','batiments','autres_immobilisations_corporelles','immobilisations_en_concession','immobilisations_en_cours','titres_mis_en_equivalence','autres_participations_creances','autres_titres_immobilises','prets_actifs_financiers_non_courants','impots_differes_actif']),
 ('total_actif_courant',['stocks_encours','clients','autres_debiteurs','impots_assimiles_actif','autres_creances_assimiles','placements_financiers_courants','tresorerie_actif']),
 ('total_general_actif',['total_actif_non_courant','total_actif_courant'])]
F_PASSIF=[('total_capitaux_propres',['capital_emis','capital_non_appele','primes_reserves','ecart_reevaluation','ecart_equivalence','resultat_net_passif','report_a_nouveau','part_societe_consolidante','part_minoritaires']),
 ('total_passifs_non_courants',['emprunts_dettes_financieres','impots_differes_provisionnes','autres_dettes_non_courantes','provisions_produits_avance']),
 ('total_passifs_courants',['fournisseurs_rattaches','impots_passif','autres_dettes','tresorerie_passif']),
 ('total_general_passif',['total_capitaux_propres','total_passifs_non_courants','total_passifs_courants'])]
TCR_CONSO=['achats_marchandises_vendues','matieres_premieres','autres_approvisionnements','variation_stocks','achats_etudes_prestations','autres_consommations','rabais_remises_obtenus_achats','sous_traitance_generale','locations','entretien_reparations','primes_assurances','personnel_exterieur','remuneration_intermediaires','publicite','deplacements_missions','autres_services','rabais_remises_obtenus_services']
def _sum0(d,keys,suf):
    vals=[d.get(f'{k}_{suf}') for k in keys]
    if all(v is None for v in vals): return None
    return sum(v or 0 for v in vals)
def _signed(t,key,suf):
    c=t.get(f'{key}_credit_{suf}'); d=t.get(f'{key}_debit_{suf}')
    if c is None and d is None: return None
    return (c or 0)-(d or 0)
def _ss(t,keys,suf):
    vals=[_signed(t,k,suf) for k in keys]
    if all(v is None for v in vals): return None
    return sum(v or 0 for v in vals)
def compute_controles(res):
    anoms={}; a,p,t=res.get('ACTIF') or {},res.get('PASSIF') or {},res.get('TCR') or {}
    def chk(tab,key,suf,ext,calc):
        if ext is None or calc is None: return
        if abs(ext-calc)>TOL: anoms[f'{tab}.{key}.{suf}']=f'extrait={ext:,.0f} calculé={calc:,.0f} écart={ext-calc:,.0f}'
    for suf in ('n','n1'):
        for k in SCHEMAS['ACTIF']['postes']:
            b,am,n=a.get(f'{k}_brut'),a.get(f'{k}_amort'),a.get(f'{k}_{suf}')
            if suf=='n' and b is not None and n is not None: chk('ACTIF',k,suf,n,b-(am or 0))
        for tot,comp in F_ACTIF: chk('ACTIF',tot,suf,a.get(f'{tot}_{suf}'),_sum0(a,comp,suf))
        for tot,comp in F_PASSIF: chk('PASSIF',tot,suf,p.get(f'{tot}_{suf}'),_sum0(p,comp,suf))
        chk('PASSIF','total_general_passif',suf,p.get(f'total_general_passif_{suf}'),a.get(f'total_general_actif_{suf}'))
        chk('TCR','chiffre_affaires_net',suf,_signed(t,'chiffre_affaires_net',suf),_ss(t,['ventes_marchandises','produits_fabriques','prestations_services','ventes_travaux','produits_annexes','rabais_remises_ristournes_accordes'],suf))
        chk('TCR','production_exercice',suf,_signed(t,'production_exercice',suf),_ss(t,['chiffre_affaires_net','production_stockee_destockee','production_immobilisee','subvention_exploitation'],suf))
        chk('TCR','consommations_exercice',suf,_signed(t,'consommations_exercice',suf),_ss(t,TCR_CONSO,suf))
        chk('TCR','valeur_ajoutee_exploitation',suf,_signed(t,'valeur_ajoutee_exploitation',suf),_ss(t,['production_exercice','consommations_exercice'],suf))
        chk('TCR','excedent_brut_exploitation',suf,_signed(t,'excedent_brut_exploitation',suf),_ss(t,['valeur_ajoutee_exploitation','charges_personnel','impots_taxes_assimiles'],suf))
        chk('TCR','resultat_operationnel',suf,_signed(t,'resultat_operationnel',suf),_ss(t,['excedent_brut_exploitation','autres_produits_operationnels','autres_charges_operationnelles','dotations_amortissements','provisions','pertes_valeur','reprises_pertes_valeur_provisions'],suf))
        chk('TCR','resultat_financier',suf,_signed(t,'resultat_financier',suf),_ss(t,['produits_financiers','charges_financieres'],suf))
        chk('TCR','resultat_ordinaire',suf,_signed(t,'resultat_ordinaire',suf),_ss(t,['resultat_operationnel','resultat_financier'],suf))
        chk('TCR','resultat_net_exercice',suf,_signed(t,'resultat_net_exercice',suf),_ss(t,['resultat_ordinaire','elements_extraordinaires_produits','elements_extraordinaires_charges','impots_exigibles_resultats','impots_differes_resultats'],suf))
        rn_p=p.get(f'resultat_net_passif_{suf}')
        chk('CROISE','resultat_net',suf,rn_p,_signed(t,'resultat_net_exercice',suf))
    return anoms
print('✅ Contrôles OK')

In [ ]:
COLORS={'META':'FFD6E4F0','ACTIF':'FFE2EFDA','PASSIF':'FFDAE3F3','TCR':'FFFCE4D6','DECL':'FFE4DFEC'}
HDRS={'META':'FF1F4E79','ACTIF':'FF375623','PASSIF':'FF203864','TCR':'FF833C00','DECL':'FF4B2D73'}
TOT_ACT=['total_actif_non_courant','total_actif_courant','total_general_actif']
def build_cols():
    cols=[('META',k,lab) for k,lab in [('fichier','Fichier'),('entreprise','Entreprise'),('nif','NIF'),('exercice','Exercice'),('annee_exercice','Année exercice'),('annee_depot','Année dépôt'),('date_traitement','Date traitement'),('temps_total_s','Temps (s)'),('tokens_total','Tokens'),('pages_trouvees','Tableaux'),('anomalies','Anomalies')]]
    for k in SCHEMAS['ACTIF']['postes']: cols.append(('ACTIF',k,k))
    for k in TOT_ACT: cols+= [('ACTIF',k+'_brut',k+' [brut]'),('ACTIF',k+'_amort',k+' [amort]')]
    for k in SCHEMAS['PASSIF']['postes']: cols.append(('PASSIF',k,k))
    for k in SCHEMAS['TCR']['postes']: cols.append(('TCR',k,k))
    for k in SCHEMAS['DECL']['postes']: cols.append(('DECL',k,k))
    return cols
def val_for(d,g,key,suf):
    if g=='DECL': return (d.get('DECL') or {}).get(key)
    t=d.get(g) or {}
    if g=='ACTIF' and (key.endswith('_brut') or key.endswith('_amort')): return t.get(key) if suf=='n' else None
    if g=='TCR':
        c=t.get(f'{key}_credit_{suf}'); dd=t.get(f'{key}_debit_{suf}')
        if c is None and dd is None: return None
        return (c or 0)-(dd or 0)
    return t.get(f'{key}_{suf}')
def create_excel(path,rows):
    all_cols=build_cols(); wb=Workbook(); ws=wb.active; ws.title='Bilans'
    grp=defaultdict(list); idx=1
    for g,_,_ in all_cols: grp[g].append(idx); idx+=1
    for g,cs in grp.items():
        s,e=cs[0],cs[-1]
        if s<e: ws.merge_cells(start_row=1,start_column=s,end_row=1,end_column=e)
        c=ws.cell(row=1,column=s); c.value=g; c.font=Font(bold=True,color='FFFFFFFF',name='Arial',size=11); c.fill=PatternFill('solid',start_color=HDRS[g]); c.alignment=Alignment(horizontal='center',vertical='center')
    for i,(g,_,lab) in enumerate(all_cols,start=1):
        c=ws.cell(row=2,column=i); c.value=lab; c.font=Font(bold=True,name='Arial',size=8); c.fill=PatternFill('solid',start_color=COLORS[g]); c.alignment=Alignment(horizontal='center',vertical='center',wrap_text=True); ws.column_dimensions[get_column_letter(i)].width=16
    ws.row_dimensions[2].height=30; ws.freeze_panes=ws.cell(row=3,column=7)
    rn=3
    for d in rows:
        an=norm_str(d.get('exercice')); an1=str(int(an)-1) if an and an.isdigit() else None
        for exer,annee,suf in [('N',an,'n'),('N-1',an1,'n1')]:
            for ci,(g,key,_) in enumerate(all_cols,start=1):
                if g=='META': val=exer if key=='exercice' else (annee if key=='annee_exercice' else d.get(key))
                else: val=val_for(d,g,key,suf)
                c=ws.cell(row=rn,column=ci); c.value=val; c.font=Font(name='Arial',size=9); c.fill=PatternFill('solid',start_color=COLORS[g])
                if isinstance(val,float): c.number_format='#,##0.00'
                if g=='META' and key=='anomalies' and val: c.font=Font(name='Arial',size=9,bold=True,color='FFCC0000')
            rn+=1
    wb.save(path); print(f'✅ Excel récap: {path} | {len(rows)} bilans')
print('✅ Export Excel OK')

In [ ]:
def log(msg):
    ligne=f"{datetime.now().strftime('%Y-%m-%d %H:%M:%S')} — {msg}"
    print(ligne)
    with open(LOG_PATH,'a',encoding='utf-8') as f: f.write(ligne+'\n')
print('✅ Log OK')

In [ ]:
deja={f.stem for f in JSON_DIR.glob('*.json')}
a_traiter=[p for p in pdfs if p.stem not in deja]
log(f'À traiter: {len(a_traiter)} | déjà: {len(deja)}')
t_total=time.time(); n_ok=n_err=0
TYPES={'ACTIF','PASSIF','TCR'}
def parse_type(text):
    t=(text or '').upper()
    for k in ('ACTIF','PASSIF','TCR','DECL'):
        if k in t: return k
    return 'AUTRE'
for num,pdf_path in enumerate(a_traiter,1):
    t0=time.time()
    try:
        pages=pdf_to_pages(pdf_path); actives=[p for p in pages if not is_blank(p['image'])]
        tok_in=tok_out=0
        mini=[resize(p['image'],600) for p in actives]; reps1=[]
        for bs in range(0,len(mini),16): reps1+=ask_batch(PROMPT_CLASSIF,mini[bs:bs+16])
        tok_in+=sum(r['tokens_in'] for r in reps1); tok_out+=sum(r['tokens_out'] for r in reps1)
        utiles=[(p,parse_type(r['text'])) for p,r in zip(actives,reps1) if parse_type(r['text'])!='AUTRE']
        parsed,decl_data,annee_depot=[],None,None
        for bs in range(0,len(utiles),GPU_BATCH_SIZE):
            batch=utiles[bs:bs+GPU_BATCH_SIZE]; reps=ask_batch(PROMPT_BILAN,[p['image'] for p,_ in batch])
            for (page,tpage),rep in zip(batch,reps):
                tok_in+=rep['tokens_in']; tok_out+=rep['tokens_out']
                data=parse_json(rep['text']); t=data.get('type',tpage)
                if t not in TYPES and t!='DECL': continue
                meta={k:data.get(k) for k in ('entreprise','nif')}; meta['exercice']=None if t=='DECL' else data.get('exercice')
                if t=='DECL':
                    decl_data=data; dep=norm_annee4(data.get('annee_souscription'))
                    if dep and not annee_depot: annee_depot=dep
                parsed.append({'index':page['index'],'type':t,'meta':meta,'data':data})
            gc.collect(); torch.cuda.empty_cache()
        ref_ent,ref_nif,ref_annee,anomalies,excluded=controle_coherence(parsed,ANNEE_ATTENDUE)
        ent_raw=next((norm_str(p['meta'].get('entreprise')) for p in parsed if p['index'] not in excluded and meme_entreprise(ref_ent or '',norm_identite(p['meta'].get('entreprise')))),None)
        tables,tcr_pages,doublons={},[],[]
        for p in parsed:
            if p['index'] in excluded or p['type']=='DECL': continue
            t=p['type']
            if t=='TCR': tcr_pages.append(p['data'])
            else:
                if t in tables: doublons.append(t); continue
                tables[t]=normalise_table(t,p['data'])
        tcr={}
        for d in tcr_pages:
            for k,v in normalise_table('TCR',d).items():
                if tcr.get(k) is None and v is not None: tcr[k]=v
        if tcr: tables['TCR']=tcr
        if doublons: anomalies.append('DOUBLON: '+', '.join(doublons))
        decl_norm=normalise_decl(decl_data) if decl_data else {}
        anoms=compute_controles({'ACTIF':tables.get('ACTIF',{}),'PASSIF':tables.get('PASSIF',{}),'TCR':tables.get('TCR',{})})
        anomalies+=list(anoms.values())
        dt=round(time.time()-t0,2); manquants=sorted(TYPES-set(tables.keys()))
        result={'fichier':pdf_path.name,'entreprise':ent_raw,'nif':ref_nif,'exercice':ref_annee,'annee_depot':annee_depot,'date_traitement':datetime.now().strftime('%Y-%m-%d %H:%M:%S'),'temps_total_s':dt,'tokens_in':tok_in,'tokens_out':tok_out,'tokens_total':tok_in+tok_out,'pages_trouvees':', '.join(sorted(tables.keys())),'anomalies':' | '.join(anomalies) if anomalies else None,'DECL':decl_norm,**{t:tables.get(t,{}) for t in TYPES}}
        with open(JSON_DIR/f'{pdf_path.stem}.json','w',encoding='utf-8') as f: json.dump(result,f,ensure_ascii=False,indent=2,default=str)
        n_ok+=1; eta=(time.time()-t_total)/num*(len(a_traiter)-num)
        msg=f'[{num:>4}/{len(a_traiter)}] ✅ {pdf_path.name} | {dt:.1f}s | {result["pages_trouvees"]}'
        if manquants: msg+=f' | ⚠️ manquants: {", ".join(manquants)}'
        if result['anomalies']: msg+=f' | 🔴 {result["anomalies"]}'
        log(msg+f' | ETA {eta/3600:.1f}h')
    except Exception as e:
        n_err+=1; log(f'[{num:>4}/{len(a_traiter)}] ❌ {pdf_path.name} — {e}'); continue
log('Génération Excel...')
rows=[json.load(open(jf,encoding='utf-8')) for jf in sorted(JSON_DIR.glob('*.json'))]
create_excel(EXCEL_PATH,rows)
log(f'✅ Terminé {time.time()-t_total:.1f}s | OK {n_ok} | Err {n_err}')

In [ ]:
dd_path=OUTPUT_DIR/'data_dictionary_bilans_v9.csv'
with open(dd_path,'w',newline='',encoding='utf-8') as f:
    w=csv.writer(f); w.writerow(['table','cle','libelle_imprime','colonnes'])
    for t,spec in SCHEMAS.items():
        for k,lab in spec['postes'].items(): w.writerow([t,k,lab,'|'.join(spec['cols'])])
print(f'✅ Data dictionary: {dd_path}')

In [ ]:
# ── Détection dynamique des coordonnées (repli si échec) ──
def norm_label(s):
    s=unicodedata.normalize('NFKD',str(s))
    s=''.join(ch for ch in s if not unicodedata.combining(ch))
    return re.sub(r'[^a-z0-9]','',s.lower())
def label_map(ws,cols=(1,2,3),max_row=80):
    m={}
    for r in range(1,max_row+1):
        for c in cols:
            v=ws.cell(row=r,column=c).value
            if v is not None: m.setdefault(norm_label(v),r)
    return m
def detect_actif_cols(ws):
    for r in range(1,15):
        vals=[(c,norm_label(ws.cell(row=r,column=c).value)) for c in range(1,15)]
        br=[c for c,v in vals if 'bruts' in v]; am=[c for c,v in vals if 'amort' in v]; ne=[c for c,v in vals if v.startswith('net')]
        if br and am and ne:
            return {'brut':br[0],'amort':am[0],'net':ne[0],'brut_n1':br[1] if len(br)>1 else None,'amort_n1':am[1] if len(am)>1 else None,'net_n1':ne[1] if len(ne)>1 else None}
    return None
def detect_passif_cols(ws):
    for r in range(1,10):
        dcols=[c for c in range(1,10) if ws.cell(row=r,column=c).value is not None and re.search(r'\d{2}/\d{4}',str(ws.cell(row=r,column=c).value))]
        if len(dcols)>=2: return dcols[0],dcols[1]
        c2=[c for c in range(1,10) if 'n-2' in norm_label(ws.cell(row=r,column=c).value)]
        if c2: return c2[0]+1,c2[0]+2
    return None
def detect_tcr_cols(ws):
    best=None
    for r in range(1,10):
        vals=[(c,norm_label(ws.cell(row=r,column=c).value)) for c in range(1,12)]
        deb=[c for c,v in vals if v.startswith('debit')]; cre=[c for c,v in vals if v.startswith('credit')]
        if len(deb)>=2 and len(cre)>=2:
            rowstr=' '.join(str(ws.cell(row=r,column=c).value) for c in range(1,12)).upper()
            best={'deb_n':deb[0],'cred_n':cre[0],'deb_n1':deb[1],'cred_n1':cre[1]}
            if 'KDZD' in rowstr: return best
    return best
RED=Font(color='FFFF0000',bold=True); REDFILL=PatternFill('solid',start_color='FFFFC7CE')
def to_kdzd(v): return None if v is None else round(v/1000.0,2)
print('✅ Détection coords OK')

In [ ]:
def generate_forfait(res):
    if not TEMPLATE_FORFAIT.exists(): print(f'❌ Template absent: {TEMPLATE_FORFAIT}'); return None
    anoms=compute_controles(res)
    ent=res.get('entreprise') or 'ENTREPRISE'; annee=res.get('exercice') or 'XXXX'
    out=FORFAIT_OUTPUT_DIR/f'FORFAIT Cas 1_{ent}_{annee}.xlsx'
    shutil.copy2(TEMPLATE_FORFAIT,out)
    wb=load_workbook(out)
    a,p,t=res.get('ACTIF') or {},res.get('PASSIF') or {},res.get('TCR') or {}
    n_written=0
    # ACTIF
    if 'Saisie actif' in wb.sheetnames:
        ws=wb['Saisie actif']; lm=label_map(ws); cols=detect_actif_cols(ws) or {'brut':2,'amort':3,'net':4,'brut_n1':5,'amort_n1':6,'net_n1':7}
        for key,r in {'ecarts_acquisition_goodwill':7,'immobilisations_incorporelles':8,'terrains':10,'batiments':11,'autres_immobilisations_corporelles':12,'immobilisations_en_concession':13,'immobilisations_en_cours':14,'titres_mis_en_equivalence':16,'autres_participations_creances':17,'autres_titres_immobilises':18,'prets_actifs_financiers_non_courants':19,'impots_differes_actif':20,'total_actif_non_courant':21,'stocks_encours':23,'clients':25,'autres_debiteurs':26,'impots_assimiles_actif':27,'autres_creances_assimiles':28,'placements_financiers_courants':30,'tresorerie_actif':31,'total_actif_courant':32,'total_general_actif':33}.items():
            rr=lm.get(norm_label(key.replace('_',' '))) or r
            for colattr,suf in [('brut','brut'),('amort','amort'),('net','n'),('net_n1','n1')]:
                src = a.get(f'{key}_{"n" if colattr=="net" else ("n1" if colattr=="net_n1" else colattr)}')
                if src is None: continue
                col={'brut':cols['brut'],'amort':cols['amort'],'net':cols['net'],'net_n1':cols.get('net_n1')}[colattr]
                if col is None: continue
                c=ws.cell(row=rr,column=col,value=to_kdzd(src)); n_written+=1
                if f'ACTIF.{key}.{"n" if colattr in ("net",) else ("n1" if colattr=="net_n1" else "n")}' in anoms: c.font=RED; c.fill=REDFILL
    # PASSIF
    if 'Saisie passif' in wb.sheetnames:
        ws=wb['Saisie passif']; lm=label_map(ws); pc=detect_passif_cols(ws) or (2,3)
        for key,r in {'capital_emis':3,'capital_non_appele':4,'primes_reserves':5,'ecart_reevaluation':6,'ecart_equivalence':7,'resultat_net_passif':8,'report_a_nouveau':9,'part_societe_consolidante':10,'part_minoritaires':11,'total_capitaux_propres':12,'emprunts_dettes_financieres':14,'impots_differes_provisionnes':15,'autres_dettes_non_courantes':16,'provisions_produits_avance':17,'total_passifs_non_courants':18,'fournisseurs_rattaches':20,'impots_passif':21,'autres_dettes':22,'tresorerie_passif':23,'total_passifs_courants':24,'total_general_passif':25}.items():
            rr=lm.get(norm_label(key.replace('_',' '))) or r
            for suf,col in [('n',pc[0]),('n1',pc[1])]:
                src=p.get(f'{key}_{suf}')
                if src is None: continue
                c=ws.cell(row=rr,column=col,value=to_kdzd(src)); n_written+=1
                if f'PASSIF.{key}.{suf}' in anoms or f'CROISE.resultat_net.{suf}' in anoms: c.font=RED; c.fill=REDFILL
    # TCR
    if 'Saisie TCR' in wb.sheetnames:
        ws=wb['Saisie TCR']; lm=label_map(ws,cols=(1,2)); tc=detect_tcr_cols(ws) or {'deb_n':3,'cred_n':4,'deb_n1':5,'cred_n1':6}
        for key,r in {'ventes_marchandises':2,'produits_fabriques':3,'prestations_services':4,'ventes_travaux':5,'produits_annexes':6,'rabais_remises_ristournes_accordes':7,'chiffre_affaires_net':8,'production_stockee_destockee':9,'production_immobilisee':10,'subvention_exploitation':11,'production_exercice':12,'achats_marchandises_vendues':13,'matieres_premieres':14,'autres_approvisionnements':15,'variation_stocks':16,'achats_etudes_prestations':17,'autres_consommations':18,'rabais_remises_obtenus_achats':19,'sous_traitance_generale':20,'locations':21,'entretien_reparations':22,'primes_assurances':23,'personnel_exterieur':24,'remuneration_intermediaires':25,'publicite':26,'deplacements_missions':27,'autres_services':28,'rabais_remises_obtenus_services':29,'consommations_exercice':30,'valeur_ajoutee_exploitation':31,'charges_personnel':32,'impots_taxes_assimiles':33,'excedent_brut_exploitation':34,'autres_produits_operationnels':35,'autres_charges_operationnelles':36,'dotations_amortissements':37,'provisions':38,'pertes_valeur':39,'reprises_pertes_valeur_provisions':40,'resultat_operationnel':41,'produits_financiers':42,'charges_financieres':43,'resultat_financier':44,'resultat_ordinaire':45,'elements_extraordinaires_produits':46,'elements_extraordinaires_charges':47,'resultat_extraordinaire':48,'impots_exigibles_resultats':49,'impots_differes_resultats':50,'resultat_net_exercice':51}.items():
            rr=lm.get(norm_label(key.replace('_',' '))) or r
            for suf,cd,cc in [('n',tc['deb_n'],tc['cred_n']),('n1',tc['deb_n1'],tc['cred_n1'])]:
                d=t.get(f'{key}_debit_{suf}'); cr=t.get(f'{key}_credit_{suf}')
                if d is not None: c=ws.cell(row=rr,column=cd,value=to_kdzd(d)); n_written+=1; (c.font:=RED,c.fill:=REDFILL) if f'TCR.{key}.{suf}' in anoms else None
                if cr is not None: c=ws.cell(row=rr,column=cc,value=to_kdzd(cr)); n_written+=1; (c.font:=RED,c.fill:=REDFILL) if f'TCR.{key}.{suf}' in anoms else None
    wb.save(out); wb.close()
    print(f'✅ FORFAIT: {out.name} | {n_written} cellules | {len(anoms)} incohérences rouges')
    return out
print('✅ Générateur FORFAIT OK')

In [ ]:
json_files=sorted(JSON_DIR.glob('*.json'))
if not json_files: print('⚠️ Aucun JSON — lance Cellule 11')
else:
    n_gen=0
    for jf in json_files:
        try:
            d=json.load(open(jf,encoding='utf-8'))
            if not (d.get('ACTIF') or d.get('PASSIF') or d.get('TCR')): continue
            if generate_forfait(d): n_gen+=1
        except Exception as e: print(f'❌ {jf.stem}: {e}')
    print(f'✅ {n_gen} FORFAIT .xlsx générés → {FORFAIT_OUTPUT_DIR}')